In [1]:
import numpy as np
import os
import bs4
import time
import json
from typing import List, Dict
from dotenv import load_dotenv
import scipy.spatial.distance as spdist
from tqdm import tqdm

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from operator import itemgetter

# LangChain imports
from langsmith import Client
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

/Users/amily/Desktop/assignment-1-evelynh037/genai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


# RAGs + Analysis

In [2]:
# Load environment variables from .env file
load_dotenv()

# Get API keys from environment variables
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# Check if API keys are set
if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY not found in environment variables.\n"
        "Please create a .env file in the project root with:\n"
        "OPENAI_API_KEY=your_openai_api_key_here"
    )

if not PINECONE_API_KEY:
    raise ValueError(
        "PINECONE_API_KEY not found in environment variables.\n"
        "Please create a .env file in the project root with:\n"
        "PINECONE_API_KEY=your_pinecone_api_key_here"
    )

# Set environment variables explicitly (in case load_dotenv didn't work)
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

print("✓ API keys loaded successfully")


✓ API keys loaded successfully


In [3]:
# Initialize embeddings wrapper (using HuggingFace model, not OpenAI)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

# Initialize Pinecone vector store
# Note: PineconeVectorStore uses the Pinecone API key, not OpenAI
index= "stream-embeddings"
text_field = "id"
vectorstore = PineconeVectorStore(  
    index_name="stream-embeddings",
    embedding=embeddings,
    text_key=text_field,
    pinecone_api_key=PINECONE_API_KEY,
)  
time.sleep(10)


/var/folders/bf/mf9fw9915q98qdc0k4pv6gnr0000gn/T/ipykernel_40087/1630984998.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [4]:
query = "What happened between Shane and Ilya during their rookie MLH season?"
similar_vectors = vectorstore.similarity_search(query, k=10)
similar_vectors

[Document(id='t_301507_0', metadata={'episode_number': 1.0, 'title': 'Heated Rivalry', 'type': 'show', 'year': '2025'}, page_content='t_301507_0'),
 Document(id='t_301507_1', metadata={'episode_number': 2.0, 'title': 'Heated Rivalry', 'type': 'show', 'year': '2025'}, page_content='t_301507_1'),
 Document(id='t_301507_4', metadata={'episode_number': 5.0, 'title': 'Heated Rivalry', 'type': 'show', 'year': '2025'}, page_content='t_301507_4'),
 Document(id='t_301507_3', metadata={'episode_number': 4.0, 'title': 'Heated Rivalry', 'type': 'show', 'year': '2025'}, page_content='t_301507_3'),
 Document(id='t_301507_5', metadata={'episode_number': 6.0, 'title': 'Heated Rivalry', 'type': 'show', 'year': '2025'}, page_content='t_301507_5'),
 Document(id='t_271053_4', metadata={'episode_number': 5.0, 'title': 'I Love LA', 'type': 'show', 'year': '2025'}, page_content='t_271053_4'),
 Document(id='t_243861_4', metadata={'episode_number': 5.0, 'title': 'The Vince Staples Show', 'type': 'show', 'year'

## Basic RAG
- query -> vector db search -> LLM -> output

In [5]:
with open("../chunk_data/merged_chunks.json", "r", encoding="utf-8") as f:
    data = json.load(f)


raw_text = {}

# read in original text
for i in data:
    raw_text[i["id"]] = i["text"]

In [6]:
llm = ChatOpenAI(model="gpt-4o-mini", seed=0)

# create custom retriever
class CustomRetriever(BaseRetriever):
    vectorstore: PineconeVectorStore
    splits: Dict[str, str]

    def _get_relevant_documents(self, query):
        docs = self.vectorstore.similarity_search(query, k=10)
        outputs = []
        for doc in docs:
            raw_text = self.splits[str(doc.id)]
            outputs.append(Document(page_content=raw_text, metadata={"id": doc.id, "title": doc.metadata["title"]}))
        return outputs
    
retriever = CustomRetriever(vectorstore=vectorstore, splits=raw_text)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    
#prompt = hub.pull("rlm/rag-prompt")
client = Client()
prompt = client.pull_prompt("rlm/rag-prompt")
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [7]:
query = "What make Benny Scanlon heatbreak in Overcompensating?"

In [8]:
answer = rag_chain.invoke(query)
answer

'Benny Scanlon experiences heartbreak in "Overcompensating" due to his struggles with self-doubt and the anxiety of trying to impress others, particularly his father. This is compounded by the pressure to perform well in softball, leading to a series of events that exacerbate his fears and insecurities. Ultimately, his emotional turmoil contributes to a challenging experience on and off the field.'

## RAG with HyDE-style query rewriting

In [9]:
# Define customed HyDE query rewriting function
def hyde_query_rewrite(input_dict: Dict[str, str]):
    original_query = input_dict["original_query"]
    rewrite_prompt = f"""
    Rewrite this question as a short plot-focused statement suitable for vector search.
    Keep character names, time period, key events, causes and results. Do not summarize emotionally. Do notadd in additional information.
    Question: "{original_query}"
"""
    rewritten_query = (llm | StrOutputParser()).invoke(rewrite_prompt)
    return rewritten_query

rag_chain_rewrite = (
    {
        # Rewrite query first (HyDE)
        "hyde_query": {"original_query": RunnablePassthrough()} 
                      | RunnableLambda(hyde_query_rewrite),

        # Retrieve and format docs using rewritten query
        "context": RunnableLambda(lambda hyde_query: format_docs(
            retriever._get_relevant_documents(hyde_query)
        )),

        # Pass original question to prompt
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()           # generate final answer
)

In [10]:
answer = rag_chain_rewrite.invoke(query)
answer

'Benny Scanlon experiences heartbreak in "Overcompensating" due to the immense pressure he feels to perform and succeed, particularly in light of his father\'s expectations. His struggles, compounded by Laurie\'s anxiety and her injuries during the championship game, lead to heightened emotions and personal turmoil. The situation showcases the impact of external pressures and personal insecurities on relationships and self-worth.'

## RAG with HyDE-style query rewriting and rerank

In [11]:
# Reranker function
def rerank_chunks(input_dict: Dict[str, list], title_weight=0.2):
    """
    Rerank retrieved chunks by combining cosine similarity with metadata match.
    
    input_dict: {
        "hyde_query": str,
        "retrieved_chunks": list of Document objects
    }
    """
    chunks = input_dict["retrieved_chunks"]
    scored_chunks = []
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )
    query_emb = embeddings.embed_query(input_dict["hyde_query"])

    for doc in chunks:
        # Ensure doc is actually a Document
        if not isinstance(doc, Document):
            continue

        chunk_emb = embeddings.embed_query(doc.page_content)

        # cosine similarity
        cos_score = sum(q * c for q, c in zip(query_emb, chunk_emb)) / (
            (sum(q*q for q in query_emb) ** 0.5) * (sum(c*c for c in chunk_emb) ** 0.5)
        )

        # metadata/title boost
        title_score = 0
        if "title" in doc.metadata and doc.metadata["title"].lower() in input_dict["hyde_query"].lower():
            title_score = title_weight

        final_score = cos_score + title_score
        scored_chunks.append((final_score, doc))

    # sort descending
    scored_chunks.sort(reverse=True, key=lambda x: x[0])
    return [doc for score, doc in scored_chunks][:5]


# RAG chain using parallel + passthrough
rag_chain_rewrite_rerank = (
    # query rewriting
    {"hyde_query": {"original_query": RunnablePassthrough()} 
                   | RunnableLambda(hyde_query_rewrite)}
    # rerank and format
    | RunnableLambda(
        lambda outputs: {
            "context": format_docs(
                rerank_chunks({
                    "hyde_query": outputs["hyde_query"],
                    "retrieved_chunks": retriever._get_relevant_documents(outputs["hyde_query"])
                })
            ),
            "question": outputs["hyde_query"]
        }
    )
    | prompt
    | llm
    | StrOutputParser()   
)

In [12]:
answer = rag_chain_rewrite_rerank.invoke(query)
answer

'In "Overcompensating," Benny Scanlon faces heartbreak after a failed romantic relationship, affecting his self-image and causing him to spiral emotionally. His struggles with self-acceptance lead him to sabotage his connections, particularly as he navigates the complexities of entering gay life. The story reflects his journey of coping while seeking to understand his identity and relationships.'

## Response analysis

In [13]:
# Define queries
queries = [
    "What make Benny Scanlon heatbreak in Overcompensating?",
    "How do Milo Bradford and his wife respond to the kidnapping?",
    "What is the name of the chef that gordon ramsay met for Māori cuisine?",
    "How do Lucy’s feelings about her breakup with Stephen and her guilt toward Evan affect her actions in this episode?",
    "Why Ilya ignored Shane's text?"
]

# Method names
methods = [
    "ChatGPT Only",
    "Basic RAG",
    "Query Rewrite (HyDE)",
    "Query Rewrite + Rerank"
]

# Initialize results storage
results = []

print("Running queries through all methods...")
print("=" * 80)

# Process each query
for query_idx, query in enumerate(tqdm(queries, desc="Processing queries")):
    print(f"\nQuery {query_idx + 1}: {query}")
    print("-" * 80)
    
    query_results = {"Query": f"Query {query_idx + 1}", "Query Text": query}
    
    # Method 1: ChatGPT Only (no RAG)
    try:
        print(f"  Running: {methods[0]}...")
        chatgpt_only_prompt = f"Answer the following question based on your knowledge: {query}"
        chatgpt_only_answer = (llm | StrOutputParser()).invoke(chatgpt_only_prompt)
        query_results[methods[0]] = chatgpt_only_answer
        print(f"  ✓ {methods[0]} completed")
    except Exception as e:
        query_results[methods[0]] = f"Error: {str(e)}"
        print(f"  ✗ {methods[0]} failed: {str(e)}")
    
    # Method 2: Basic RAG
    try:
        print(f"  Running: {methods[1]}...")
        basic_rag_answer = rag_chain.invoke(query)
        query_results[methods[1]] = basic_rag_answer
        print(f"  ✓ {methods[1]} completed")
    except Exception as e:
        query_results[methods[1]] = f"Error: {str(e)}"
        print(f"  ✗ {methods[1]} failed: {str(e)}")
    
    # Method 3: Query Rewrite (HyDE)
    try:
        print(f"  Running: {methods[2]}...")
        rewrite_answer = rag_chain_rewrite.invoke(query)
        query_results[methods[2]] = rewrite_answer
        print(f"  ✓ {methods[2]} completed")
    except Exception as e:
        query_results[methods[2]] = f"Error: {str(e)}"
        print(f"  ✗ {methods[2]} failed: {str(e)}")
    
    # Method 4: Query Rewrite + Rerank
    try:
        print(f"  Running: {methods[3]}...")
        rerank_answer = rag_chain_rewrite_rerank.invoke(query)
        query_results[methods[3]] = rerank_answer
        print(f"  ✓ {methods[3]} completed")
    except Exception as e:
        query_results[methods[3]] = f"Error: {str(e)}"
        print(f"  ✗ {methods[3]} failed: {str(e)}")
    
    results.append(query_results)
    print()

print("=" * 80)
print("All queries processed!")

# Create DataFrame
df_results = pd.DataFrame(results)

# Display DataFrame
print("\nResults DataFrame:")
print("=" * 80)
df_results

Running queries through all methods...


Processing queries:   0%|          | 0/5 [00:00<?, ?it/s]


Query 1: What make Benny Scanlon heatbreak in Overcompensating?
--------------------------------------------------------------------------------
  Running: ChatGPT Only...
  ✓ ChatGPT Only completed
  Running: Basic RAG...
  ✓ Basic RAG completed
  Running: Query Rewrite (HyDE)...
  ✓ Query Rewrite (HyDE) completed
  Running: Query Rewrite + Rerank...


Processing queries:  20%|██        | 1/5 [00:12<00:50, 12.57s/it]

  ✓ Query Rewrite + Rerank completed


Query 2: How do Milo Bradford and his wife respond to the kidnapping?
--------------------------------------------------------------------------------
  Running: ChatGPT Only...
  ✓ ChatGPT Only completed
  Running: Basic RAG...
  ✓ Basic RAG completed
  Running: Query Rewrite (HyDE)...
  ✓ Query Rewrite (HyDE) completed
  Running: Query Rewrite + Rerank...


...

Processing queries:  80%|████████  | 4/5 [00:46<00:11, 11.81s/it]

  ✓ Query Rewrite + Rerank completed


Query 5: Why Ilya ignored Shane's text?
--------------------------------------------------------------------------------
  Running: ChatGPT Only...
  ✓ ChatGPT Only completed
  Running: Basic RAG...
  ✓ Basic RAG completed
  Running: Query Rewrite (HyDE)...
  ✓ Query Rewrite (HyDE) completed
  Running: Query Rewrite + Rerank...


Processing queries: 100%|██████████| 5/5 [00:59<00:00, 11.83s/it]

  ✓ Query Rewrite + Rerank completed

All queries processed!

Results DataFrame:


,Query,Query Text,ChatGPT Only,Basic RAG,Query Rewrite (HyDE),Query Rewrite + Rerank
0,Query 1,What make Benny Scanlon heatbreak in Overcompe...,"In ""Overcompensating,"" a novel by Benny Scanlo...","Benny Scanlon experiences heartbreak in ""Overc...","Benny Scanlon experiences heartbreak in ""Overc...","In ""Overcompensating,"" Benny Scanlon faces hea..."
1,Query 2,How do Milo Bradford and his wife respond to t...,"I'm sorry, but I don't have specific informati...","Milo Bradford and his wife, Marissa, respond t...","Milo Bradford and his wife, Marissa, respond t...",Milo Bradford and his wife face the urgency of...
2,Query 3,What is the name of the chef that gordon ramsa...,Gordon Ramsay met chef *Andrew Clarke* for Māo...,The chef that Gordon Ramsay met for Māori cuis...,The chef Gordon Ramsay met for Māori cuisine i...,Gordon Ramsay meets chef Monique Fiso in the S...
3,Query 4,How do Lucy’s feelings about her breakup with ...,"In the episode, Lucy's feelings about her brea...",Lucy feels conflicted about her breakup with S...,Lucy’s feelings about her breakup with Stephen...,"In this episode, Lucy grapples with her guilt ..."
4,Query 5,Why Ilya ignored Shane's text?,There could be many reasons why Ilya ignored S...,Ilya ignored Shane's texts following his team'...,Ilya ignored Shane's texts after his Russian h...,"In the modern-day scenario, Ilya fails to resp..."


In [14]:
# Display detailed comparison table
print("=" * 80)
print("DETAILED ANSWER COMPARISON")
print("=" * 80)

for idx, row in df_results.iterrows():
    print(f"\n{'='*80}")
    print(f"QUERY {idx + 1}: {row['Query Text']}")
    print(f"{'='*80}\n")
    
    for method in methods:
        answer = row[method]
        # Truncate very long answers for display
        display_answer = str(answer)
        if len(display_answer) > 500:
            display_answer = display_answer[:500] + "... [truncated]"
        print(f"{method}:")
        print(f"  {display_answer}")
        print(f"  Length: {len(str(answer))} characters")
        print()

# Save results to CSV for further analysis
df_results.to_csv('rag_comparison_results.csv', index=False)
print("\n" + "=" * 80)
print("Results saved to 'rag_comparison_results.csv'")
print("=" * 80)

DETAILED ANSWER COMPARISON

QUERY 1: What make Benny Scanlon heatbreak in Overcompensating?

ChatGPT Only:
  In "Overcompensating," a novel by Benny Scanlon, the protagonist experiences heartbreak primarily due to personal relationships and challenges related to his own self-identity. The narrative explores themes of love, loss, and the quest for validation. Benny's heartbreak often stems from misunderstandings and the pressure he feels to meet the expectations of those around him, leading to emotional turmoil. The story captures the intricacies of human connections and the vulnerability that comes wit... [truncated]
  Length: 629 characters

Basic RAG:
  Benny Scanlon experiences heartbreak in "Overcompensating" primarily due to Laurie's struggles with anxiety and her feelings of inadequacy in softball, which affect her relationship with her father, Coach Dan. As she tries to impress him, her failure on the field and the injury to Kai compound her self-doubt. Ultimately, Laurie consid

Evaluation of Pipeline Performance

1. LLM Only
- Unable to answer effectively when the question is out-of-scope.
- Generates lengthy, off-topic responses that rely on guesswork or generalizations.
- Often hallucinates plot details or misses the main entities entirely.

2. Basic RAG
- Answers are grounded in retrieved documents, so they are related to the plot.
- Responses tend to be summaries of retrieved content rather than directly answering the query.
- Less verbose than LLM-only and more on-topic.

3. Query Rewrite (HyDE)
- Incorporates instructions that guide the answer toward the specific intention of the query.
- Summaries are tailored to address the question, rather than just repeating the retrieved content.
- Ensures the response is relevant and entailed by the query.
	
3. Query Rewrite + Rerank
- Ensures retrieval prioritizes the most relevant documents, weighting by metadata (e.g., title, entity tags).
- Reduces interference from peripheral or misleading chunks.
- Answers align closely with main entities and key plot events, producing concise, accurate, and focused responses.

